In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Loading data... please wait")

filepath = r"C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\data\accepted_2007_to_2018Q4.csv.gz"

df = pd.read_csv(filepath, compression='gzip', low_memory=False)

print(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loading data... please wait
Data loaded: 2,260,701 rows, 151 columns


In [2]:
# Check what we're starting with
print("Before filtering:")
print(df['loan_status'].value_counts())
print(f"Total rows: {len(df):,}\n")

# Keep only rows where loan outcome is known and clear
keep_status = ['Fully Paid', 'Charged Off', 'Default']

df = df[df['loan_status'].isin(keep_status)]

# Create binary target: 1 = default/bad, 0 = fully paid/good
df['target'] = df['loan_status'].apply(
    lambda x: 1 if x in ['Charged Off', 'Default'] else 0
)

print("After filtering:")
print(df['loan_status'].value_counts())
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nDefault rate: {df['target'].mean()*100:.2f}%")
print(f"Remaining rows: {len(df):,}")

Before filtering:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64
Total rows: 2,260,701

After filtering:
loan_status
Fully Paid     1076751
Charged Off     268559
Default             40
Name: count, dtype: int64

Target distribution:
target
0    1076751
1     268599
Name: count, dtype: int64

Default rate: 19.96%
Remaining rows: 1,345,350


In [5]:
# These columns only exist AFTER the loan has been issued
# A bank would NOT have this information at application time
# Including them would be data leakage

leaky_columns = [
    # Payment history - doesn't exist at application time
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
    'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
    
    # Outstanding amounts - only known after loan starts
    'out_prncp', 'out_prncp_inv',
    
    # Updated credit scores - these change after loan is issued
    'last_fico_range_high', 'last_fico_range_low',
    
    # Other post-loan tracking columns
    'next_pymnt_d', 'last_credit_pull_d',
    
    # The original loan status - we've already extracted our target from this
    'loan_status'
]

print(f"Columns before dropping leaky columns: {df.shape[1]}")

df = df.drop(columns=leaky_columns)

print(f"Columns after dropping leaky columns: {df.shape[1]}")
print(f"Dropped {len(leaky_columns)} leaky columns")

Columns before dropping leaky columns: 152
Columns after dropping leaky columns: 136
Dropped 16 leaky columns


In [7]:
# These columns carry no predictive information
useless_columns = [
    'id',              # Just a unique identifier number
    'member_id',       # 100% missing anyway
    'url',             # Web link to loan page
    'desc',            # Free text, mostly empty
    'title',           # Borrower's own description (very noisy)
    'zip_code',        # Too granular, and partially masked anyway
    'pymnt_plan',      # Almost always 'n'
    'policy_code',     # Always 1, no variation
]

print(f"Columns before: {df.shape[1]}")

# Only drop columns that actually exist in our dataframe
# (some might already be gone from the leaky drop)
cols_to_drop = [c for c in useless_columns if c in df.columns]
df = df.drop(columns=cols_to_drop)

print(f"Columns after: {df.shape[1]}")
print(f"Dropped: {cols_to_drop}")

Columns before: 136
Columns after: 128
Dropped: ['id', 'member_id', 'url', 'desc', 'title', 'zip_code', 'pymnt_plan', 'policy_code']


In [9]:
# Calculate missing percentage for all remaining columns
missing_pct = (df.isnull().sum() / len(df)) * 100

# Identify columns with more than 50% missing
high_missing = missing_pct[missing_pct > 50]

print(f"Columns with more than 50% missing values: {len(high_missing)}")
print("\nThese columns will be dropped:")
print(high_missing.sort_values(ascending=False).to_string())

# Drop them
df = df.drop(columns=high_missing.index)

print(f"\nColumns remaining: {df.shape[1]}")
print(f"Rows remaining: {df.shape[0]:,}")

Columns with more than 50% missing values: 55

These columns will be dropped:
orig_projected_additional_accrued_interest   99.72
hardship_end_date                            99.57
hardship_dpd                                 99.57
hardship_reason                              99.57
hardship_status                              99.57
deferral_term                                99.57
hardship_amount                              99.57
hardship_start_date                          99.57
hardship_type                                99.57
payment_plan_start_date                      99.57
hardship_length                              99.57
hardship_last_payment_amount                 99.57
hardship_payoff_balance_amount               99.57
hardship_loan_status                         99.57
sec_app_mths_since_last_major_derog          99.51
sec_app_revol_util                           98.64
revol_bal_joint                              98.61
sec_app_earliest_cr_line                     98.61
sec_

In [11]:
# We identified three problematic columns in our exploration
# annual_inc max = $110 million (clearly an error)
# dti max = 999 and min = -1 (impossible values)
# revol_util max = 892% (extreme outlier)

print("Before outlier treatment:")
print(f"annual_inc max: ${df['annual_inc'].max():,.0f}")
print(f"dti max: {df['dti'].max()}, dti min: {df['dti'].min()}")
print(f"revol_util max: {df['revol_util'].max()}")

# Cap annual income at 99th percentile
# This preserves high earners but removes impossible values
annual_inc_cap = df['annual_inc'].quantile(0.99)
df['annual_inc'] = df['annual_inc'].clip(upper=annual_inc_cap)

# Remove rows where dti is negative or above 100
# A dti above 100 means debt is more than income - extremely rare
# and likely a data error at these extreme levels
df = df[df['dti'] >= 0]
df = df[df['dti'] <= 100]

# Cap revol_util at 100%
# Above 100% is technically possible but extreme values are likely errors
df['revol_util'] = df['revol_util'].clip(upper=100)

print("\nAfter outlier treatment:")
print(f"annual_inc max: ${df['annual_inc'].max():,.0f}")
print(f"dti max: {df['dti'].max()}, dti min: {df['dti'].min()}")
print(f"revol_util max: {df['revol_util'].max()}")
print(f"\nRows remaining: {len(df):,}")

Before outlier treatment:
annual_inc max: $10,999,200
dti max: 999.0, dti min: -1.0
revol_util max: 892.3

After outlier treatment:
annual_inc max: $250,000
dti max: 100.0, dti min: 0.0
revol_util max: 100.0

Rows remaining: 1,344,441


In [13]:
print("=" * 50)
print("CLEANING PROGRESS SUMMARY")
print("=" * 50)
print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nDefault rate: {df['target'].mean()*100:.2f}%")

# How much missing data is left?
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"\nColumns still with missing values: {len(remaining_missing)}")
print("\nTop 15 remaining missing columns:")
print(((remaining_missing / len(df)) * 100)
      .sort_values(ascending=False)
      .head(15)
      .round(2)
      .to_string())

CLEANING PROGRESS SUMMARY
Rows:    1,344,441
Columns: 73

Target distribution:
target
0    1076056
1     268385
Name: count, dtype: int64

Default rate: 19.96%

Columns still with missing values: 42

Top 15 remaining missing columns:
mths_since_recent_inq   12.94
num_tl_120dpd_2m         8.73
mo_sin_old_il_acct       7.85
emp_title                6.33
emp_length               5.79
pct_tl_nvr_dlq           5.03
avg_cur_bal              5.02
mo_sin_old_rev_tl_op     5.02
mo_sin_rcnt_rev_tl_op    5.02
num_rev_accts            5.02
num_tl_op_past_12m       5.02
num_tl_90g_dpd_24m       5.02
num_tl_30dpd             5.02
num_rev_tl_bal_gt_0      5.02
num_accts_ever_120_pd    5.02


In [15]:
# Let's see what columns we have left and understand them
print("Remaining columns:")
print("=" * 60)

for i, col in enumerate(df.columns):
    missing_pct = df[col].isnull().mean() * 100
    dtype = df[col].dtype
    print(f"{i+1:>3}. {col:<45} | {str(dtype):<10} | {missing_pct:.1f}% missing")

Remaining columns:
  1. loan_amnt                                     | float64    | 0.0% missing
  2. funded_amnt                                   | float64    | 0.0% missing
  3. funded_amnt_inv                               | float64    | 0.0% missing
  4. term                                          | object     | 0.0% missing
  5. int_rate                                      | float64    | 0.0% missing
  6. installment                                   | float64    | 0.0% missing
  7. grade                                         | object     | 0.0% missing
  8. sub_grade                                     | object     | 0.0% missing
  9. emp_title                                     | object     | 6.3% missing
 10. emp_length                                    | object     | 5.8% missing
 11. home_ownership                                | object     | 0.0% missing
 12. annual_inc                                    | float64    | 0.0% missing
 13. verification_status         

In [17]:
# Drop columns we've decided are unnecessary, duplicate, or leaky
cols_to_drop = [
    'funded_amnt',          # duplicate of loan_amnt
    'funded_amnt_inv',      # duplicate of loan_amnt
    'debt_settlement_flag', # post-loan leaky flag
    'hardship_flag',        # nearly constant, no predictive value
    'disbursement_method',  # operational detail, not predictive
    'initial_list_status',  # internal Lending Club detail
    'application_type',     # almost all individual after joint drop
    'addr_state',           # geographic, too granular
    'emp_title',            # free text, too noisy
    'issue_d',              # date we don't need
]

print(f"Columns before: {df.shape[1]}")
df = df.drop(columns=cols_to_drop)
print(f"Columns after: {df.shape[1]}")

Columns before: 73
Columns after: 63


In [19]:
# 'term' is currently " 36 months" or " 60 months" (note the space)
print("Before conversion:")
print(df['term'].value_counts())

# Extract just the number
df['term'] = df['term'].str.extract('(\d+)').astype(int)

print("\nAfter conversion:")
print(df['term'].value_counts())

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\DELL G3\AppData\Local\Temp\ipykernel_5172\1741376557.py:6: SyntaxWarning: invalid escape sequence '\d'
  df['term'] = df['term'].str.extract('(\d+)').astype(int)


Before conversion:
term
36 months    1020225
60 months     324216
Name: count, dtype: int64

After conversion:
term
36    1020225
60     324216
Name: count, dtype: int64


In [21]:
# emp_length has values like "10+ years", "3 years", "< 1 year", and NaN
print("Before conversion:")
print(df['emp_length'].value_counts(dropna=False).head(15))

# Define a mapping dictionary
emp_length_map = {
    '< 1 year': 0,
    '1 year':   1,
    '2 years':  2,
    '3 years':  3,
    '4 years':  4,
    '5 years':  5,
    '6 years':  6,
    '7 years':  7,
    '8 years':  8,
    '9 years':  9,
    '10+ years': 10
}

df['emp_length'] = df['emp_length'].map(emp_length_map)

print("\nAfter conversion:")
print(df['emp_length'].value_counts(dropna=False))

Before conversion:
emp_length
10+ years    442165
2 years      121718
< 1 year     107945
3 years      107578
1 year        88473
5 years       84137
4 years       80547
NaN           77901
6 years       62727
8 years       60702
7 years       59615
9 years       50933
Name: count, dtype: int64

After conversion:
emp_length
10.00    442165
2.00     121718
0.00     107945
3.00     107578
1.00      88473
5.00      84137
4.00      80547
NaN       77901
6.00      62727
8.00      60702
7.00      59615
9.00      50933
Name: count, dtype: int64


In [23]:
# earliest_cr_line looks like "Aug-2003", "Dec-1999" etc.
# We want to know: how many years of credit history does this person have?
# We calculate: year loan was issued - year credit history started

print("Sample values:")
print(df['earliest_cr_line'].head(10))

# Convert to datetime
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# Calculate credit history in years
# We use 2018 as reference point (last year of our dataset)
reference_year = 2018
df['credit_history_years'] = reference_year - df['earliest_cr_line'].dt.year

# Drop the original date column now that we've extracted what we need
df = df.drop(columns=['earliest_cr_line'])

print("\nCredit history years - summary:")
print(df['credit_history_years'].describe())

Sample values:
0     Aug-2003
1     Dec-1999
2     Aug-2000
4     Jun-1998
5     Oct-1987
6     Jun-1990
7     Feb-1999
8     Apr-2002
9     Nov-1994
12    Jun-1996
Name: earliest_cr_line, dtype: object

Credit history years - summary:
count   1344441.00
mean         19.31
std           7.61
min           3.00
25%          14.00
50%          18.00
75%          23.00
max          84.00
Name: credit_history_years, dtype: float64


In [25]:
# fico_range_low and fico_range_high together represent one score
# Average them into a single clean feature
df['fico_score'] = (df['fico_range_low'] + df['fico_range_high']) / 2

# Drop the originals
df = df.drop(columns=['fico_range_low', 'fico_range_high'])

print("FICO score - summary:")
print(df['fico_score'].describe())

FICO score - summary:
count   1344441.00
mean        698.18
std          31.85
min         627.00
25%         672.00
50%         692.00
75%         712.00
max         847.50
Name: fico_score, dtype: float64


In [27]:
# Check what categorical columns remain
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Remaining categorical columns:")
for col in cat_cols:
    print(f"  {col}: {df[col].nunique()} unique values")
    print(f"    {df[col].value_counts().head(5).to_dict()}")
    print()

Remaining categorical columns:
  grade: 7 unique values
    {'B': 392532, 'C': 381401, 'A': 235010, 'D': 200784, 'E': 93570}

  sub_grade: 35 unique values
    {'C1': 85435, 'B4': 83155, 'B5': 82489, 'B3': 81787, 'C2': 79157}

  home_ownership: 6 unique values
    {'MORTGAGE': 664988, 'RENT': 534213, 'OWN': 144763, 'ANY': 286, 'OTHER': 144}

  verification_status: 3 unique values
    {'Source Verified': 521201, 'Verified': 418097, 'Not Verified': 405143}

  purpose: 14 unique values
    {'debt_consolidation': 779802, 'credit_card': 295125, 'home_improvement': 87426, 'other': 77808, 'major_purchase': 29413}



In [29]:
# Drop sub_grade - redundant with grade
df = df.drop(columns=['sub_grade'])

# Clean up home_ownership rare categories
df['home_ownership'] = df['home_ownership'].replace({
    'ANY': 'OTHER',
    'NONE': 'OTHER'
})

print("home_ownership after cleanup:")
print(df['home_ownership'].value_counts())
print(f"\nColumns remaining: {df.shape[1]}")

home_ownership after cleanup:
home_ownership
MORTGAGE    664988
RENT        534213
OWN         144763
OTHER          477
Name: count, dtype: int64

Columns remaining: 61


In [31]:
# Grade has a natural order - A is best, G is worst
# We encode this as numbers preserving that order
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}

print("Before encoding:")
print(df['grade'].value_counts())

df['grade'] = df['grade'].map(grade_map)

print("\nAfter encoding:")
print(df['grade'].value_counts().sort_index())

Before encoding:
grade
B    392532
C    381401
A    235010
D    200784
E     93570
F     32032
G      9112
Name: count, dtype: int64

After encoding:
grade
1    235010
2    392532
3    381401
4    200784
5     93570
6     32032
7      9112
Name: count, dtype: int64


In [33]:
# Columns to one-hot encode
ohe_columns = ['home_ownership', 'verification_status', 'purpose']

print(f"Columns before one-hot encoding: {df.shape[1]}")

# drop_first=False because we want all categories visible
# prefix ensures column names are clear e.g. purpose_debt_consolidation
df = pd.get_dummies(df, columns=ohe_columns, drop_first=False)

print(f"Columns after one-hot encoding: {df.shape[1]}")

# Show the new columns created
new_cols = [c for c in df.columns if any(
    c.startswith(prefix) for prefix in ohe_columns
)]
print(f"\nNew columns created ({len(new_cols)}):")
for col in new_cols:
    print(f"  {col}")

Columns before one-hot encoding: 61
Columns after one-hot encoding: 79

New columns created (21):
  home_ownership_MORTGAGE
  home_ownership_OTHER
  home_ownership_OWN
  home_ownership_RENT
  verification_status_Not Verified
  verification_status_Source Verified
  verification_status_Verified
  purpose_car
  purpose_credit_card
  purpose_debt_consolidation
  purpose_educational
  purpose_home_improvement
  purpose_house
  purpose_major_purchase
  purpose_medical
  purpose_moving
  purpose_other
  purpose_renewable_energy
  purpose_small_business
  purpose_vacation
  purpose_wedding


In [35]:
print("=" * 55)
print("PRE-IMPUTATION SUMMARY")
print("=" * 55)
print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")

# Check data types
print(f"\nData types:")
print(df.dtypes.value_counts())

# Check remaining missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nColumns still needing imputation: {len(missing)}")
print("\nMissing value counts:")
print(missing.to_string())

# Confirm target is still intact
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"Default rate: {df['target'].mean()*100:.2f}%")

PRE-IMPUTATION SUMMARY
Rows:    1,344,441
Columns: 79

Data types:
float64    54
bool       21
int32       2
int64       2
Name: count, dtype: int64

Columns still needing imputation: 41

Missing value counts:
mths_since_recent_inq         173945
num_tl_120dpd_2m              117403
mo_sin_old_il_acct            105540
emp_length                     77901
pct_tl_nvr_dlq                 67681
avg_cur_bal                    67549
num_rev_accts                  67528
mo_sin_old_rev_tl_op           67528
mo_sin_rcnt_rev_tl_op          67528
num_tl_op_past_12m             67527
num_tl_90g_dpd_24m             67527
num_tl_30dpd                   67527
num_accts_ever_120_pd          67527
num_op_rev_tl                  67527
tot_hi_cred_lim                67527
num_il_tl                      67527
num_bc_tl                      67527
num_actv_rev_tl                67527
num_rev_tl_bal_gt_0            67527
num_actv_bc_tl                 67527
total_rev_hi_lim               67527
mo_sin_rcnt_t

In [37]:
# These columns have so few missing values that simple median is fine
tiny_missing_cols = [
    'revol_util', 'pub_rec_bankruptcies', 'chargeoff_within_12_mths',
    'collections_12_mths_ex_med', 'tax_liens', 'inq_last_6mths'
]

for col in tiny_missing_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled {col} with median: {median_val}")

print(f"\nMissing values remaining: {df.isnull().sum().sum():,}")

Filled revol_util with median: 52.2
Filled pub_rec_bankruptcies with median: 0.0
Filled chargeoff_within_12_mths with median: 0.0
Filled collections_12_mths_ex_med with median: 0.0
Filled tax_liens with median: 0.0
Filled inq_last_6mths with median: 0.0

Missing values remaining: 2,438,616


In [39]:
# Get all columns that still have missing values
still_missing = df.columns[df.isnull().any()].tolist()
print(f"Columns to impute: {len(still_missing)}")
print(still_missing)

# Calculate median for each column within each grade group
# Then fill missing values using those grade-specific medians
print("\nImputing using grade-stratified medians...")

for col in still_missing:
    # Calculate median per grade
    grade_medians = df.groupby('grade')[col].transform('median')
    
    # Fill missing values with grade-specific median
    df[col] = df[col].fillna(grade_medians)

# Check if any missing values remain
# (can happen if an entire grade has no values for a column)
remaining = df.isnull().sum().sum()
print(f"\nMissing values after grade-stratified imputation: {remaining:,}")

if remaining > 0:
    # Fall back to overall median for anything still missing
    df = df.fillna(df.median(numeric_only=True))
    print(f"Missing values after fallback median: {df.isnull().sum().sum():,}")

Columns to impute: 35
['emp_length', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit']

Imputing using grade-stratified medians...

Missing values after grade-stratified imputation: 0


In [41]:
print("=" * 55)
print("FINAL CLEANED DATASET SUMMARY")
print("=" * 55)
print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")

print(f"\nMissing values: {df.isnull().sum().sum()}")

print(f"\nData types:")
print(df.dtypes.value_counts())

print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"Default rate: {df['target'].mean()*100:.2f}%")

print(f"\nSample of final dataset:")
df.head(3)

FINAL CLEANED DATASET SUMMARY
Rows:    1,344,441
Columns: 79

Missing values: 0

Data types:
float64    54
bool       21
int32       2
int64       2
Name: count, dtype: int64

Target distribution:
target
0    1076056
1     268385
Name: count, dtype: int64
Default rate: 19.96%

Sample of final dataset:


,loan_amnt,term,int_rate,installment,grade,emp_length,annual_inc,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_inq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,target,credit_history_years,fico_score,home_ownership_MORTGAGE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,purpose_car,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,3600.00,36,13.99,123.03,3,10.00,55000.00,5.91,0.00,1.00,7.00,0.00,2765.00,29.70,13.00,0.00,0.00,722.00,144904.00,9300.00,4.00,20701.00,1506.00,37.20,0.00,0.00,148.00,128.00,3.00,3.00,1.00,4.00,4.00,2.00,2.00,4.00,2.00,5.00,3.00,4.00,9.00,4.00,7.00,0.00,0.00,0.00,3.00,76.90,0.00,0.00,0.00,178050.00,7746.00,2400.00,13734.00,0,15,677.00,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
1,24700.00,36,11.99,820.28,3,10.00,65000.00,16.06,1.00,4.00,22.00,0.00,21470.00,19.20,38.00,0.00,0.00,0.00,204396.00,111800.00,4.00,9733.00,57830.00,27.10,0.00,0.00,113.00,192.00,2.00,2.00,4.00,2.00,0.00,0.00,5.00,5.00,13.00,17.00,6.00,20.00,27.00,5.00,22.00,0.00,0.00,0.00,2.00,97.40,7.70,0.00,0.00,314017.00,39475.00,79300.00,24667.00,0,19,717.00,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,20000.00,60,10.78,432.66,2,10.00,63000.00,10.78,0.00,0.00,6.00,0.00,7869.00,56.20,18.00,0.00,0.00,0.00,189699.00,14000.00,6.00,31617.00,2737.00,55.90,0.00,0.00,125.00,184.00,14.00,14.00,5.00,101.00,10.00,0.00,2.00,3.00,2.00,4.00,6.00,4.00,7.00,3.00,6.00,0.00,0.00,0.00,0.00,100.00,50.00,0.00,0.00,218418.00,18696.00,6200.00,14877.00,0,18,697.00,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False


In [45]:
# Save to the outputs folder so we never have to run cleaning again
# We save as parquet - much faster to read than CSV for large files
output_path = r"C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\outputs\loans_cleaned.parquet"

df.to_parquet(output_path, index=False)

print(f"Saved cleaned dataset to:")
print(output_path)
print(f"\nFile contains {len(df):,} rows and {df.shape[1]} columns")
print("You can now load this file in future notebooks instead of reprocessing everything")

Saved cleaned dataset to:
C:\Users\DELL G3\Desktop\RMIT SEM - 4\Loan Club Project\outputs\loans_cleaned.parquet

File contains 1,344,441 rows and 79 columns
You can now load this file in future notebooks instead of reprocessing everything


In [47]:
try:
    import xgboost as xgb
    print(f"XGBoost installed. Version: {xgb.__version__}")
except ImportError:
    print("Not installed. Run in Anaconda terminal: conda install -c conda-forge xgboost")

Not installed. Run in Anaconda terminal: conda install -c conda-forge xgboost


In [1]:
import numpy as np
import xgboost as xgb
import shap
import joblib

print(f"numpy:   {np.__version__}")
print(f"xgboost: {xgb.__version__}")
print(f"shap:    {shap.__version__}")
print("\nAll imports successful!")

numpy:   1.26.4
xgboost: 3.2.0
shap:    0.46.0

All imports successful!
